## Object-Oriented QUBO Solver Tutorial (high-level façade)

This notebook demonstrates the **object-oriented** (“happy path”) usage of `qubo-solver`: you solve a QUBO end-to-end through a single, ergonomic façade (e.g. `Solver(...).solve()`), with behavior driven by clear configuration objects rather than manually wiring each pipeline step.

The goal is twofold:

- **Ergonomics:** provide a fast, readable entry point that is easy to demo and easy to adopt—define the QUBO, choose a configuration, call `solve()`, and get back a well-formed `Solution`.
- **Policy centralization:** keep cross-cutting concerns (defaults, seeding/reproducibility, dtype/precision choices, backend selection, and recommended “standard” combinations of algorithms) in one place, so users don’t have to remember how to assemble a correct pipeline.

Under the hood, the façade is intended to remain **thin**: it should call the same underlying functional building blocks (transforms → embedding → drive shaping → solvers), while exposing a simpler top-level interface for most users.

## Setup and Imports

In [ ]:
from __future__ import annotations

import itertools
import numpy as np
import random
import torch
from typing import Literal
import warnings

import qoolqit

from qubosolver import (
    Instance,
    Solution,
    SingleSolution,
    Analyzer,
    Solver,
    SolverConfig,
    EmbeddingConfig,
    DriveShapingConfig,
    ClassicalConfig,
    ClassicalSolverType,
    EmbedderType,
    DriveType,
    torch_rng,
    bitstrings,
    vectori,
    linalg,
    tensor,
)

## Utility Functions

In [ ]:
def gather_optimal_solutions(solutions: Solution) -> list[SingleSolution]:
    """Find all solutions with minimum cost."""
    min_cost = solutions[0].cost
    return [d for d in solutions if np.allclose(d.cost, min_cost)]

def manual_seed(seed: int) -> torch.Generator:
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    return torch_rng(seed)

def interaction_matrix_from_vertices(vertices: torch.Tensor) -> torch.Tensor:
    """Create interaction matrix based on vertex distances."""
    U = 1.0 / torch.cdist(vertices, vertices) ** 6
    U.fill_diagonal_(0.0)
    return U

## Creating a Simple QUBO Problem

We'll create a simple QUBO instance based on geometric vertices with interaction strengths.

In [ ]:
def create_simple_qubo():
    """Create a simple QUBO problem and find its optimal solutions."""
    
    # Define vertices in 2D space
    sqrt3 = np.sqrt(3.0)
    vertices = tensor.tensor([
        [0.0, 0.0],
        [-1.0, 0.0],
        [-1.5, -0.5 * sqrt3],
        [-0.5, -0.5 * sqrt3],
        [4.0, 0.0],
    ])
    
    n_qubits = vertices.shape[0]
    
    # Create QUBO matrix from vertex interactions
    diagonal_scale = -2.0
    diagonal = torch.ones(n_qubits, dtype=linalg.dtype())
    Q = interaction_matrix_from_vertices(vertices) + diagonal_scale * torch.diag(diagonal)
    Q /= Q.max()  # Normalize
    
    # Find optimal solutions by exhaustive search
    solutions = Solution()
    solutions.bitstrings = bitstrings.tensor(list(itertools.product([0, 1], repeat=n_qubits)))
    solutions.counts = vectori.zeros(solutions.bitstrings.shape[0]).fill_(1)
    solutions.compute_costs(Q).sort_by_cost().compute_probabilities()
    
    expected_optimal_solutions = gather_optimal_solutions(solutions)
    
    print(f"QUBO Matrix Shape: {Q.shape}")
    print(f"Expected Minimum cost: {expected_optimal_solutions[0].cost:.6f}")
    print(f"Number of optimal solutions: {len(expected_optimal_solutions)}")
    print(f"Optimal bitstrings: {[s.string for s in expected_optimal_solutions]}")
    
    return Instance(matrix=Q), expected_optimal_solutions

# Create the QUBO problem
seed = 16844214
manual_seed(seed)
qubo, expected_optimal_solutions = create_simple_qubo()

QUBO Matrix Shape: torch.Size([5, 5])
Expected Minimum cost: -5.925370
Number of optimal solutions: 1
Optimal bitstrings: ['10101']


## Solution Analysis Function

In [ ]:
def analyze_solution(solutions: Solution, expected_optimal_solutions: list[SingleSolution], 
                    expect_optimality: bool = True):
    """Analyze and validate QUBO solutions."""
    
    # Check for duplicate solutions
    unique_count = solutions.bitstrings.unique(dim=0).shape[0]
    total_count = len(solutions)
    print(f"Solutions are unique: {unique_count == total_count}")
    
    # Create analyzer for detailed statistics
    analyzer = Analyzer([solutions])
    print(f"\nSolution Statistics:\n{analyzer.df}")
    
    # Find optimal solutions
    optimal_solutions = gather_optimal_solutions(solutions)
    min_cost = optimal_solutions[0].cost
    
    print(f"\nFound minimum cost: {min_cost:.6f}")
    print(f"Found optimal bitstrings: {[s.string for s in optimal_solutions]}")
    print(f"Number of found optimal solutions: {len(optimal_solutions)}")
    
    if expect_optimality:
        # Check if we found the true optimum
        expected_cost = expected_optimal_solutions[0].cost
        cost_match = np.allclose(min_cost, expected_cost)
        print("\nOptimality check:")
        print(f"Expected cost: {expected_cost:.6f}")
        print(f"Cost matches expected: {cost_match}")
        
        # Check solution quality
        cumulated_probability = sum(s.probability for s in optimal_solutions)
        print(f"Total probability of optimal solutions: {cumulated_probability:.4f}")
        print(f"Good solution quality (>75%): {cumulated_probability > 0.75}")
    
    return optimal_solutions

## Quantum Solving Workflow

Now let's solve the QUBO problem using quantum methods with different configurations.

In [ ]:
"""Solve QUBO using quantum methods."""

embedding_method: Literal["blade", "greedy"] = "blade"
drive_shaping_method: Literal["heuristic", "optimized"] = "heuristic"
preprocessing: bool = True
postprocessing: bool = True

seed = 16844214
manual_seed(seed)

warnings.filterwarnings("ignore")

print("\n=== Quantum Solving ===")
print(f"Embedding: {embedding_method}, Drive shaping: {drive_shaping_method}")
print(f"Preprocessing: {preprocessing}, Postprocessing: {postprocessing}")

if embedding_method == "blade":
    embedding_config = EmbeddingConfig(embedding_method=EmbedderType.BLADE, min_distance=1.001)
elif embedding_method == "greedy":
    embedding_config = EmbeddingConfig(
        embedding_method=EmbedderType.GREEDY,
        min_distance=1.001,
        greedy_traps=100,
        greedy_spacing=0.1,
    )
else:
    raise ValueError(f"Invalid embedding method: {embedding_method}")

if drive_shaping_method == "optimized":
    drive_shaping_config = DriveShapingConfig(
        drive_shaping_method=DriveType.OPTIMIZED,
        optimized_n_calls=11,
        optimized_seed=seed,
        dmm=False,
    )
elif drive_shaping_method == "heuristic":
    drive_shaping_config = DriveShapingConfig(
        drive_shaping_method=DriveType.HEURISTIC, heuristic_kappa=0.25, dmm=False
    )
else:
    raise ValueError(f"Invalid drive shaping method: {drive_shaping_method}")

config = SolverConfig(
    use_quantum=True,
    embedding=embedding_config,
    drive_shaping=drive_shaping_config,
    do_postprocessing=postprocessing,
    do_preprocessing=preprocessing,
    device=qoolqit.AnalogDevice(),
)

solver = Solver(qubo, config)
solution = solver.solve()
solution.compute_costs(qubo.matrix).sort_by_cost().compute_probabilities()
   
analyze_solution(solution, expected_optimal_solutions)


=== Quantum Solving ===
Embedding: blade, Drive shaping: heuristic
Preprocessing: True, Postprocessing: True
Solutions are unique: True

Solution Statistics:
  labels bitstrings     costs  counts  probs
0      0      10101 -5.925370     984  0.984
1      0      01001 -3.999872      11  0.011
2      0      00011 -3.999784       5  0.005

Found minimum cost: -5.925370
Found optimal bitstrings: ['10101']
Number of found optimal solutions: 1

Optimality check:
Expected cost: -5.925370
Cost matches expected: True
Total probability of optimal solutions: 0.9840
Good solution quality (>75%): True


[SingleSolution(bitstring=tensor([1, 0, 1, 0, 1], dtype=torch.int8), cost=-5.925370216369629, probability=0.984000027179718)]

## Classical Solving Methods

Let's also solve the same problem using classical optimization methods.

In [ ]:
"""Solve QUBO using classical methods."""

solving_method: Literal["cplex", "tabu", "sa", "sa+tabu", "random"] = "cplex"
preprocessing: bool = True
postprocessing: bool = True

seed = 16844214
manual_seed(seed)
    
print("\n=== Classical Solving ===")
print(f"Method: {solving_method}")
print(f"Preprocessing: {preprocessing}, Postprocessing: {postprocessing}")

classical_solvers = {
    "cplex": ClassicalSolverType.CPLEX,
    "tabu": ClassicalSolverType.TABU_SEARCH,
    "sa": ClassicalSolverType.SIMULATED_ANNEALING,
    "sa+tabu": ClassicalSolverType.SIMULATED_ANNEALING_TABU_SEARCH,
    "random": ClassicalSolverType.RANDOM,
}

classical_config = ClassicalConfig(
    classical_solver_type=classical_solvers[solving_method],
    max_bitstrings=1,
    sa_seed=seed,
)

config = SolverConfig(
    use_quantum=False,
    do_postprocessing=postprocessing,
    do_preprocessing=preprocessing,
    classical=classical_config,
)

solver = Solver(qubo, config)
solution = solver.solve()
solution.compute_costs(qubo.matrix).sort_by_cost().compute_probabilities()

expect_optimality = solving_method != "random"
analyze_solution(solution, expected_optimal_solutions, expect_optimality)


=== Classical Solving ===
Method: cplex
Preprocessing: True, Postprocessing: True
Solutions are unique: True

Solution Statistics:
  labels bitstrings    costs  counts  probs
0      0      10101 -5.92537       1    1.0

Found minimum cost: -5.925370
Found optimal bitstrings: ['10101']
Number of found optimal solutions: 1

Optimality check:
Expected cost: -5.925370
Cost matches expected: True
Total probability of optimal solutions: 1.0000
Good solution quality (>75%): True


[SingleSolution(bitstring=tensor([1, 0, 1, 0, 1], dtype=torch.int8), cost=-5.925370216369629, probability=1.0)]